In [ ]:
##import os
###os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [1]:
from transformers import pipeline,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,AutoTokenizer,AutoModelForQuestionAnswering,AutoModelForCausalLM
from datasets import load_dataset,DatasetDict
from huggingface_hub import login,notebook_login
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

In [2]:
%pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetry-proto-1.38.0:
      Successfully uninstalled opentelemetry-proto-1.38.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-

In [3]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "deepseek-r1-LoRA-smol"
print(bool(os.environ.get("WANDB_API_KEY")))  # True

True


In [4]:
secrets = UserSecretsClient()
os.environ["Hugging Face"] = secrets.get_secret("Hugging Face")
login(token=os.environ["Hugging Face"])
print(bool(os.environ.get("Hugging Face")))

True


In [ ]:
dataset=load_dataset("HuggingFaceTB/smoltalk",'self-oss-instruct')

In [ ]:
dataset

In [ ]:
dataset['train'][0]["messages"]

In [ ]:
def convert_to_chatml(batch):
    conversations=[]
    for message in batch['messages']:
        conversations.append(
            [
                {"role": "user", "content": message[0]['content']},
                {"role": "assistant", "content": message[1]['content']},
            ]
        )
    return {'messages':conversations}
        

In [ ]:
new_dataset=dataset.map(convert_to_chatml,batched=True)

In [ ]:
new_dataset

In [ ]:
new_dataset['train']['messages']

In [10]:
tokenizer=AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def apply_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=False,
            )
            for conversation in batch["messages"]
        ]
    }

In [ ]:
formatted_dataset = new_dataset.map(apply_template,batched=True)

In [ ]:
formatted_dataset['train'][0]['text']

In [ ]:
formatted_dataset.save_to_disk("/kaggle/working/formatted_dataset")

In [ ]:
%pip install -U trl

In [ ]:
%pip install -U "trl[peft]"

In [7]:
%pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%%writefile train_ddp.py
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer
import peft
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
tokenizer=AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model=AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",torch_dtype=torch.float16)
model.config.use_cache = False
args=SFTConfig("DeepSeek-R1-Distill-Qwen-1.5B-LoRA",num_train_epochs=2,per_device_train_batch_size=16, per_device_eval_batch_size=8,eval_accumulation_steps=4,gradient_accumulation_steps=4,learning_rate=2e-5,
               eval_strategy="epoch",save_strategy="epoch",logging_steps=10,fp16=True,push_to_hub=True,report_to="wandb")
rank=8
alpha=16
dropout=0.05
peft_config=LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=dropout,bias="none",target_modules=["q_proj","v_proj","k_proj","o_proj"],task_type="CAUSAL_LM")
formatted_dataset = load_from_disk("/kaggle/working/formatted_dataset")
trainer=SFTTrainer(model=model,args=args,peft_config=peft_config,train_dataset=formatted_dataset['train'],eval_dataset=formatted_dataset["test"],processing_class=tokenizer)
trainer.train()
trainer.push_to_hub()

In [ ]:
!accelerate launch \
    --multi_gpu \
    --num_processes 2 \
    --num_machines 1 \
    train_ddp.py

In [8]:
model=AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",torch_dtype=torch.float16)
from peft import PeftModel
peft_model = PeftModel.from_pretrained(model,"Adnan2942/DeepSeek-R1-Distill-Qwen-1.5B-LoRA", torch_dtype=torch.float16)
merged_model = peft_model.merge_and_unload(safe_merge=True)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


adapter_model.safetensors:   0%|          | 0.00/8.75M [00:00<?, ?B/s]

In [9]:
merged_model.push_to_hub("DeepSeek-R1-Distill-Qwen-1.5B-LoRA-Smol",max_shard_size="5GB")
tokenizer.push_to_hub("DeepSeek-R1-Distill-Qwen-1.5B-LoRA-Smol")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

NameError: name 'tokenizer' is not defined